In [73]:
import numpy as np
import pandas as pd
import joblib
import json

from itertools import product

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve

from catboost import CatBoostClassifier

from xgboost import XGBClassifier



In [2]:
X_train = pd.read_csv('../data/X_processed.csv')
X_val = pd.read_csv('../data/X_val.csv')

y_train = X_train.pop('Class')
y_val = X_val.pop('Class')

In [3]:
features = X_train.columns.tolist()

preprocessor = ColumnTransformer(
    [
        ('scaler', StandardScaler(), features),
    ],
    remainder='drop'
)

In [4]:
catboost_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', CatBoostClassifier(iterations=1000,
                                    early_stopping_rounds=50,
                                    eval_metric='PRAUC',
                                    verbose=False)),
    ]
)

xgboost_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(n_estimators=1000,
                                eval_metric='aucpr',
                                early_stopping_rounds=50)),
    ]
)

logreg_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(solver='lbfgs', max_iter=5000)),
    ]
)

random_forest_pipeline = Pipeline(
     [
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(n_jobs=-1)),
    ]
)

In [69]:
pr = clone(preprocessor)
pr.fit(X_train)

fit_kwargs = {
    'model__eval_set' : [(pr.transform(X_val), y_val)],
     'model__verbose': False
}

catboost_hp = {
    'learning_rate' : [0.01, 0.03, 0.1],  # (0, 1]
    'depth' : [4, 6, 8],                    # depth of each tree [1, 16]
    'l2_leaf_reg' : [1, 3, 10],            # Coefficient at the L2 regularization term of the cost function.
    'auto_class_weights' : ['Balanced', 'SqrtBalanced']
}

r = (y_train == 0).sum() / (y_train == 1).sum()
xgboost_hp = {
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'max_depth': [4, 6, 8],
    'reg_lambda': [0.1, 1, 10],
    'scale_pos_weight': [1, r / 2, r, r * 2],
}

logreg_hp = {
    'C' : [0.001, 0.01, 0.1, 1],
    'class_weight' : ['balanced', None]
}

random_forest_hp = {
    'n_estimators' : [1000],
    'max_depth': [10, None],
    'class_weight' : ['balanced' , None]
}

In [6]:
def tune_hyperparameters(model, hps, fit_kwargs=None):
    fit_kwargs = fit_kwargs or {}

    comparison_table = pd.DataFrame({'Train PR-AUC' : [], 'VAL PR-AUC' : [], 'Train Val diff' : []})
    keys = list(hps.keys())

    for hp in keys:
        comparison_table[hp] = None

    for values in list(product(*hps.values())):
        current_model = clone(model)
        params = {f'model__{k}' : v for (k, v) in zip(keys, values)}
        current_model.set_params(**params)

        current_model.fit(X_train, y_train, **fit_kwargs)

        y_train_pred = current_model.predict_proba(X_train)[:,1]
        y_val_pred = current_model.predict_proba(X_val)[:,1]

        train_pr_auc = average_precision_score(y_train, y_train_pred)
        val_pr_auc = average_precision_score(y_val, y_val_pred)

        row = {'Train PR-AUC' : train_pr_auc,
               'VAL PR-AUC' : val_pr_auc,
               'Train Val diff' : train_pr_auc - val_pr_auc}

        row.update(dict(zip(keys, values)))

        comparison_table.loc[len(comparison_table)] = row

    return comparison_table

In [7]:
catboost_results = tune_hyperparameters(catboost_pipeline, catboost_hp, fit_kwargs)

In [8]:
catboost_results

,Train PR-AUC,VAL PR-AUC,Train Val diff,learning_rate,depth,l2_leaf_reg,auto_class_weights
0,0.716980,0.661170,0.055810,0.01,4,1,Balanced
1,0.816953,0.731736,0.085217,0.01,4,1,SqrtBalanced
2,0.726782,0.684325,0.042457,0.01,4,3,Balanced
3,0.828543,0.737016,0.091528,0.01,4,3,SqrtBalanced
4,0.693579,0.652429,0.041150,0.01,4,10,Balanced
5,0.841703,0.736650,0.105054,0.01,4,10,SqrtBalanced
6,0.785850,0.697627,0.088224,0.01,6,1,Balanced
7,0.815885,0.744669,0.071215,0.01,6,1,SqrtBalanced
8,0.801477,0.708179,0.093298,0.01,6,3,Balanced
9,0.802811,0.728053,0.074758,0.01,6,3,SqrtBalanced


In [9]:
xgboost_results = tune_hyperparameters(xgboost_pipeline, xgboost_hp, fit_kwargs)

[0]	validation_0-aucpr:0.77532
[1]	validation_0-aucpr:0.77888
[2]	validation_0-aucpr:0.78366
[3]	validation_0-aucpr:0.78696
[4]	validation_0-aucpr:0.78913
[5]	validation_0-aucpr:0.78921
[6]	validation_0-aucpr:0.78911
[7]	validation_0-aucpr:0.78831
[8]	validation_0-aucpr:0.78587
[9]	validation_0-aucpr:0.78482
[10]	validation_0-aucpr:0.78317
[11]	validation_0-aucpr:0.78173
[12]	validation_0-aucpr:0.78559
[13]	validation_0-aucpr:0.78301
[14]	validation_0-aucpr:0.78255
[15]	validation_0-aucpr:0.78293
[16]	validation_0-aucpr:0.78290
[17]	validation_0-aucpr:0.78333
[18]	validation_0-aucpr:0.78349
[19]	validation_0-aucpr:0.78357
[20]	validation_0-aucpr:0.78377
[21]	validation_0-aucpr:0.78357
[22]	validation_0-aucpr:0.78344
[23]	validation_0-aucpr:0.78318
[24]	validation_0-aucpr:0.78311
[25]	validation_0-aucpr:0.78321
[26]	validation_0-aucpr:0.78358
[27]	validation_0-aucpr:0.78341
[28]	validation_0-aucpr:0.78429
[29]	validation_0-aucpr:0.78433
[30]	validation_0-aucpr:0.78388
[31]	validation_0-

In [10]:
xgboost_results

,Train PR-AUC,VAL PR-AUC,Train Val diff,learning_rate,max_depth,reg_lambda,scale_pos_weight
0,0.889028,0.814422,0.074606,0.01,4,0.1,1.000000
1,0.826579,0.711570,0.115009,0.01,4,0.1,297.520979
2,0.838090,0.698649,0.139440,0.01,4,0.1,595.041958
3,0.843384,0.690775,0.152610,0.01,4,0.1,1190.083916
4,0.907718,0.820720,0.086998,0.01,4,1.0,1.000000
...,...,...,...,...,...,...,...
139,1.000000,0.813719,0.186281,0.30,8,1.0,1190.083916
140,0.985230,0.828441,0.156789,0.30,8,10.0,1.000000
141,1.000000,0.824506,0.175494,0.30,8,10.0,297.520979
142,1.000000,0.824195,0.175805,0.30,8,10.0,595.041958


In [11]:
logreg_results = tune_hyperparameters(logreg_pipeline, logreg_hp)

/Users/zarubin/PycharmProjects/CreditCardFraudDetection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/zarubin/PycharmProjects/CreditCardFraudDetection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/zarubin/PycharmProjects/CreditCardFraudDetection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/zarubin/PycharmProjects/CreditCardFraudDetection/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/zarubin/PycharmProjects/CreditCardFraudDetection/.venv/lib/p

In [12]:
logreg_results

,Train PR-AUC,VAL PR-AUC,Train Val diff,C,class_weight
0,0.743095,0.634491,0.108604,0.001,balanced
1,0.769947,0.694749,0.075198,0.001,None
2,0.751685,0.672375,0.079309,0.010,balanced
3,0.777245,0.696588,0.080658,0.010,None
4,0.761930,0.677151,0.084779,0.100,balanced
5,0.780586,0.706899,0.073687,0.100,None
6,0.764724,0.678372,0.086352,1.000,balanced
7,0.780495,0.707400,0.073095,1.000,None


In [13]:
random_forest_results = tune_hyperparameters(random_forest_pipeline, random_forest_hp)

In [14]:
random_forest_results

,Train PR-AUC,VAL PR-AUC,Train Val diff,n_estimators,max_depth,class_weight
0,0.976964,0.768581,0.208383,1000,10,balanced
1,0.971881,0.806155,0.165725,1000,10,None
2,1.000000,0.813345,0.186655,1000,None,balanced
3,1.000000,0.814788,0.185212,1000,None,None


It seems like we have overfitting with the best hyperparameter set, as PR-AUC on train is close to 1. But it is also possible in a regular case, since the 0-class is prevailing, the model could perfectly learn dependence.
To check this we can explore dependence of VAL PR-AUC on max_depth. If the quality increases as depth grows, it is not overfitting.

In [15]:
catboost_results.groupby('depth')['VAL PR-AUC'].aggregate(['mean', 'max'])

,mean,max
depth,,
4,0.710752,0.774654
6,0.720853,0.776244
8,0.761438,0.816236


In [16]:
xgboost_results.groupby('max_depth')['VAL PR-AUC'].aggregate(['mean', 'max'])

,mean,max
max_depth,,
4,0.788914,0.843069
6,0.784917,0.835640
8,0.785735,0.829513


As shown above, the validation score plateaus at higher depth values rather than declining, which does not suggest additional overfitting from depth alone - though this should be confirmed by comparing train and validation curves directly. To rule out confounding from the other hyperparameters, let's vary depth with everything else fixed.

In [17]:
best_catboost = catboost_results.loc[catboost_results['VAL PR-AUC'].idxmax()]
best_xgb = xgboost_results.loc[xgboost_results['VAL PR-AUC'].idxmax()]
best_logreg = logreg_results.loc[logreg_results['VAL PR-AUC'].idxmax()]
best_rf = random_forest_results.loc[random_forest_results['VAL PR-AUC'].idxmax()]

In [18]:
catboost_tune_depth = {
    'learning_rate' : [best_catboost['learning_rate']],
    'depth' : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'l2_leaf_reg' : [best_catboost['l2_leaf_reg']],
    'auto_class_weights' : [best_catboost['auto_class_weights']]
}

xgboost_tune_depth = {
    'learning_rate': [best_xgb['learning_rate']],
    'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'reg_lambda': [best_xgb['reg_lambda']],
    'scale_pos_weight': [best_xgb['scale_pos_weight']],
}

In [19]:
catboost_depth_tuning_result = tune_hyperparameters(catboost_pipeline, catboost_tune_depth, fit_kwargs)

In [20]:
catboost_depth_tuning_result

,Train PR-AUC,VAL PR-AUC,Train Val diff,learning_rate,depth,l2_leaf_reg,auto_class_weights
0,0.698512,0.650891,0.047621,0.1,1,1,SqrtBalanced
1,0.854346,0.760909,0.093437,0.1,2,1,SqrtBalanced
2,0.864080,0.751372,0.112708,0.1,3,1,SqrtBalanced
3,0.835316,0.730313,0.105002,0.1,4,1,SqrtBalanced
4,0.833830,0.739420,0.094411,0.1,5,1,SqrtBalanced
5,0.853073,0.754032,0.099041,0.1,6,1,SqrtBalanced
6,0.939427,0.786873,0.152554,0.1,7,1,SqrtBalanced
7,1.000000,0.816236,0.183764,0.1,8,1,SqrtBalanced
8,1.000000,0.810884,0.189116,0.1,9,1,SqrtBalanced
9,1.000000,0.813434,0.186566,0.1,10,1,SqrtBalanced


In [21]:
xgboost_depth_tuning_result = tune_hyperparameters(xgboost_pipeline, xgboost_tune_depth, fit_kwargs)

[0]	validation_0-aucpr:0.02488
[1]	validation_0-aucpr:0.06031
[2]	validation_0-aucpr:0.20738
[3]	validation_0-aucpr:0.20736
[4]	validation_0-aucpr:0.59738
[5]	validation_0-aucpr:0.59882
[6]	validation_0-aucpr:0.67643
[7]	validation_0-aucpr:0.67910
[8]	validation_0-aucpr:0.67945
[9]	validation_0-aucpr:0.67199
[10]	validation_0-aucpr:0.67194
[11]	validation_0-aucpr:0.67207
[12]	validation_0-aucpr:0.67254
[13]	validation_0-aucpr:0.67222
[14]	validation_0-aucpr:0.67170
[15]	validation_0-aucpr:0.68016
[16]	validation_0-aucpr:0.68046
[17]	validation_0-aucpr:0.69052
[18]	validation_0-aucpr:0.68868
[19]	validation_0-aucpr:0.69506
[20]	validation_0-aucpr:0.69209
[21]	validation_0-aucpr:0.67922
[22]	validation_0-aucpr:0.66238
[23]	validation_0-aucpr:0.68031
[24]	validation_0-aucpr:0.70342
[25]	validation_0-aucpr:0.67470
[26]	validation_0-aucpr:0.67036
[27]	validation_0-aucpr:0.66308
[28]	validation_0-aucpr:0.66313
[29]	validation_0-aucpr:0.66303
[30]	validation_0-aucpr:0.66263
[31]	validation_0-

In [22]:
xgboost_depth_tuning_result

,Train PR-AUC,VAL PR-AUC,Train Val diff,learning_rate,max_depth,reg_lambda,scale_pos_weight
0,0.740906,0.700632,0.040274,0.3,1,1.0,1190.083916
1,0.806354,0.749590,0.056764,0.3,2,1.0,1190.083916
2,1.000000,0.820380,0.179620,0.3,3,1.0,1190.083916
3,1.000000,0.843069,0.156931,0.3,4,1.0,1190.083916
4,1.000000,0.827303,0.172697,0.3,5,1.0,1190.083916
5,1.000000,0.818416,0.181584,0.3,6,1.0,1190.083916
6,1.000000,0.816723,0.183277,0.3,7,1.0,1190.083916
7,1.000000,0.813719,0.186281,0.3,8,1.0,1190.083916
8,1.000000,0.815116,0.184884,0.3,9,1.0,1190.083916
9,1.000000,0.814507,0.185493,0.3,10,1.0,1190.083916


Consider both boostings:
- XGBoost achieves 1.0 on train even with depth=2, it suggests that it is not overfitting but model manages to distinguish small amount of frauds from other transactions. So I think it is reasonable to take boosting with depth=4 which have the maximum PR-AUC on validation.
- CatBoost have growth on validation until the highest considering depth(=10).

In [51]:
best_catboost  = catboost_depth_tuning_result.loc[catboost_depth_tuning_result['VAL PR-AUC'].idxmax()]
best_xgb = xgboost_depth_tuning_result.loc[xgboost_depth_tuning_result['VAL PR-AUC'].idxmax()]

In [52]:
series = [best_catboost, best_xgb, best_logreg, best_rf]
compare_models = pd.DataFrame({
    'model' : [model.steps[-1][1].__class__.__name__ for model in [catboost_pipeline, xgboost_pipeline, logreg_pipeline, random_forest_pipeline]],
    'train' : [s.to_dict()['Train PR-AUC'] for s in series],
    'val' : [s.to_dict()['VAL PR-AUC'] for s in series]
})

In [53]:
compare_models

,model,train,val
0,CatBoostClassifier,1.000000,0.816236
1,XGBClassifier,1.000000,0.843069
2,LogisticRegression,0.780495,0.707400
3,RandomForestClassifier,1.000000,0.814788


In [80]:
final_model = clone(xgboost_pipeline)
final_model.set_params(model__learning_rate=0.3, model__max_depth=4, model__reg_lambda=1, model__scale_pos_weight=r * 2)
final_model.fit(X_train, y_train, **fit_kwargs)

y_probs = final_model.predict_proba(X_val)[:, 1]

[0]	validation_0-aucpr:0.10866
[1]	validation_0-aucpr:0.18222
[2]	validation_0-aucpr:0.55740
[3]	validation_0-aucpr:0.55999
[4]	validation_0-aucpr:0.56376
[5]	validation_0-aucpr:0.64208
[6]	validation_0-aucpr:0.65309
[7]	validation_0-aucpr:0.66174
[8]	validation_0-aucpr:0.67746
[9]	validation_0-aucpr:0.68224
[10]	validation_0-aucpr:0.67876
[11]	validation_0-aucpr:0.68657
[12]	validation_0-aucpr:0.69234
[13]	validation_0-aucpr:0.69487
[14]	validation_0-aucpr:0.69046
[15]	validation_0-aucpr:0.68660
[16]	validation_0-aucpr:0.68178
[17]	validation_0-aucpr:0.67189
[18]	validation_0-aucpr:0.67743
[19]	validation_0-aucpr:0.67648
[20]	validation_0-aucpr:0.67469
[21]	validation_0-aucpr:0.63954
[22]	validation_0-aucpr:0.63389
[23]	validation_0-aucpr:0.68520
[24]	validation_0-aucpr:0.67497
[25]	validation_0-aucpr:0.67039
[26]	validation_0-aucpr:0.67806
[27]	validation_0-aucpr:0.67511
[28]	validation_0-aucpr:0.69401
[29]	validation_0-aucpr:0.68071
[30]	validation_0-aucpr:0.68202
[31]	validation_0-

In [81]:
def find_threshold(y_true, y_proba, min_precision):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)

    appropriate = precision[:-1] >= min_precision
    if not appropriate.any():
        return None, None

    idx = np.flatnonzero(appropriate)
    best = idx[np.argmax(recall[:-1][idx])]

    return recall[best], thresholds[best]

In [82]:
for precision in [0.5, 0.6, 0.7, 0.8]:
    recall, threshold = find_threshold(y_val, y_probs, precision)
    print(f'precision>={precision}', 'recall', recall, 'threshold', threshold)

precision>=0.5 recall 0.8484848484848485 threshold 0.004332446
precision>=0.6 recall 0.8484848484848485 threshold 0.008498336
precision>=0.7 recall 0.8282828282828283 threshold 0.014296126
precision>=0.8 recall 0.8080808080808081 threshold 0.03741361


precision>=0.7 gives best precision-recall tradeoff for this task.

In [83]:
recall, threshold = find_threshold(y_val, y_probs, 0.7)


In [84]:
joblib.dump(final_model, '../models/fraud_pipeline.joblib')

['../models/fraud_pipeline.joblib']

In [85]:
metadata = {
    'model': 'XGBClassifier',
    'hyperparameters': {
        'learning_rate': 0.3,
        'max_depth': 4,
        'reg_lambda': 1,
        'scale_pos_weight': float(r * 2),
    },
    'threshold': float(threshold),
    'val_metrics': {
        'pr_auc': float(average_precision_score(y_val, y_probs)),
        'recall_at_precision': float(recall),
    }
}
with open('../models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)